# MEIKA Vox · transcribir audios

1. Selecciona **GPU** en **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, si está disponible.
2. Pulsa ▶ en **Preparar y abrir MEIKA Vox**, justo debajo, y autoriza tu Drive.
3. El panel aparecerá **aquí mismo**: elige carpeta y audios para transcribir.


In [ ]:
#@title Preparar y abrir MEIKA Vox
import importlib.util
import subprocess
import sys
import threading
from pathlib import Path

from google.colab import drive, output
from IPython.display import clear_output

if any(t.name == "MEIKA_Vox_batch" and t.is_alive() for t in threading.enumerate()):
    raise RuntimeError("Hay un lote activo. Deténlo desde el panel o espera a que termine.")

REPO = Path("/content/meika-vox")
# La interfaz y el motor se descargan de la misma versión pública.
REPO_REF = "main"


def checked(command):
    result = subprocess.run([str(item) for item in command], capture_output=True, text=True)
    if result.returncode:
        print(result.stderr[-3000:] or result.stdout[-3000:])
        raise RuntimeError(
            "La preparación falló. Comprueba la conexión y ejecuta esta celda de nuevo."
        )


print("1/3 · Preparando FFmpeg y Git…")
checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])
print("2/3 · Descargando MEIKA Vox…")
if not (REPO / ".git").exists():
    checked(["git", "clone", "--quiet", "https://github.com/MeikaLab/meika-vox.git", REPO])
checked(["git", "-C", REPO, "fetch", "--quiet", "origin", REPO_REF])
checked(["git", "-C", REPO, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
print("3/3 · Instalando dependencias; la primera preparación puede tardar varios minutos…")
checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "-e", str(REPO) + "[whisperx]", "ipywidgets>=8,<9"])
# Si se reejecuta el cuaderno en la misma sesión, no conservar módulos de la versión anterior.
for name in list(sys.modules):
    if name == "meika_vox" or name.startswith("meika_vox."):
        del sys.modules[name]
sys.path.insert(0, str(REPO / "src"))
runtime = importlib.import_module("meika_vox.runtime").inspect_runtime()
if not runtime.asr_ready:
    raise RuntimeError(
        "La instalación quedó incompleta. Reinicia la sesión y ejecuta todo de nuevo."
    )
print("Preparación comprobada. Ahora autoriza tu Google Drive.")

drive.mount("/content/drive")
output.enable_custom_widget_manager()
spec = importlib.util.spec_from_file_location("vox_ui", REPO / "scripts/colab_ui.py")
ui = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ui)
# El panel aparece en esta primera celda; se retiran los mensajes de preparación.
clear_output(wait=True)
ui.build_panel(REPO, Path("/content/drive/MyDrive"))


**Gratis y de código abierto.** Colab procesa los audios en Google y requiere acceso a tu Drive; no se publican en GitHub. GPU y duración variables. Revisa las transcripciones antes de usar citas.
